# 📈 Bushfire Analytics Dashboard

Ties together fire risk prediction, resource allocation, and burn scar mapping into a single overview.

**Requires:** Projects 1–3 to have been run first so their outputs exist.

**Model:** deepseek-v4-pro

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Libraries loaded')

## Load All Project Data

In [ ]:
base = Path('..')
loaded = []

# Project 1 — Fire Risk Prediction
p1_csv = base / 'project-1-risk-prediction' / 'processed' / 'forest_fire_risk_clean.csv'
if p1_csv.exists():
    df1 = pd.read_csv(p1_csv)
    state_cols = [c for c in df1.columns if c.startswith('state_')]
    tenure_cols = [c for c in df1.columns if c.startswith('tenure_')]
    loaded.append(f'P1: {len(df1):,} rows')
else:
    df1 = None
    state_cols = tenure_cols = []
    loaded.append('P1: not found — run fire risk notebook first')

# Project 2 — Resource Allocation
p2_json = base / 'project-2-resource-allocation' / 'outputs' / 'resource_allocation_plan.json'
if p2_json.exists():
    with open(p2_json) as f:
        plan = json.load(f)
    df2 = pd.DataFrame(plan['allocation'])
    loaded.append(f'P2: {len(df2)} states')
else:
    df2 = None
    plan = None
    loaded.append('P2: not found')

# Project 3 — Burn Scar Mapping
p3_mask = base / 'project-3-burn-scar-mapping' / 'outputs' / 'burn_mask.npy'
p3_diff = base / 'project-3-burn-scar-mapping' / 'outputs' / 'ndvi_diff.npy'
if p3_mask.exists() and p3_diff.exists():
    burn_mask = np.load(p3_mask)
    ndvi_diff = np.load(p3_diff)
    loaded.append(f'P3: burn_mask {burn_mask.shape}, {burn_mask.mean()*100:.1f}% burned')
else:
    burn_mask = None
    ndvi_diff = None
    loaded.append('P3: not found')

print('Loaded:', ' | '.join(loaded))

## Dashboard Overview (9-Panel)

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Bushfire Analytics Dashboard — Project Overview', fontsize=16, fontweight='bold')

# 1. Fire rate by state
ax1 = fig.add_subplot(3, 3, 1)
if df1 is not None and state_cols:
    rates = []
    for sc in state_cols:
        name = sc.replace('state_', '')
        sub = df1[df1[sc] == 1]
        if len(sub):
            rates.append({'state': name, 'fire_rate': sub['unplanned_5'].mean()})
    rdf = pd.DataFrame(rates).sort_values('fire_rate')
    colors = ['#d9534f' if v > 0.15 else '#f0ad4e' if v > 0.05 else '#5cb85c' for v in rdf['fire_rate']]
    ax1.barh(rdf['state'], rdf['fire_rate'], color=colors)
    ax1.set_title('Fire Rate by State', fontweight='bold')
    ax1.set_xlabel('Rate')

# 2. Budget allocation
ax2 = fig.add_subplot(3, 3, 2)
if df2 is not None:
    ds = df2.sort_values('budget', ascending=True)
    ax2.barh(ds['state'], ds['budget'] / 1e9,
             color=sns.color_palette('Blues_r', len(ds)))
    ax2.set_title('Budget Allocation ($B)', fontweight='bold')
    ax2.set_xlabel('$ Billions')

# 3. Resource breakdown
ax3 = fig.add_subplot(3, 3, 3)
if df2 is not None:
    res = df2.melt(id_vars='state',
                   value_vars=['appliance_heavy', 'appliance_light', 'crew_member'],
                   var_name='resource', value_name='count')
    sns.barplot(data=res, x='resource', y='count', hue='state', ax=ax3, palette='Set2')
    ax3.set_title('Resource Requirements', fontweight='bold')
    ax3.tick_params(axis='x', rotation=30)

# 4. Tenure risk
ax4 = fig.add_subplot(3, 3, 4)
if df1 is not None and tenure_cols:
    tr = []
    for tc in tenure_cols:
        name = tc.replace('tenure_', '')
        sub = df1[df1[tc] == 1]
        if len(sub):
            tr.append({'tenure': name, 'fire_rate': sub['unplanned_5'].mean()})
    tdf = pd.DataFrame(tr).sort_values('fire_rate')
    colors = ['#d9534f' if v > 0.15 else '#f0ad4e' if v > 0.05 else '#5cb85c' for v in tdf['fire_rate']]
    ax4.barh(tdf['tenure'], tdf['fire_rate'], color=colors)
    ax4.set_title('Fire Risk by Tenure', fontweight='bold')
    ax4.set_xlabel('Rate')

# 5. Sensitivity
ax5 = fig.add_subplot(3, 3, 5)
if plan:
    scenarios = ['Base', '+10%', '+20%', '+30%', '+50%']
    mults = [1.0, 1.1, 1.2, 1.3, 1.5]
    budgets = [plan['total_budget_aud'] * m / 1e9 for m in mults]
    ax5.plot(scenarios, budgets, 'o-', color='#d9534f', lw=2, ms=8)
    ax5.fill_between(range(len(scenarios)), budgets, alpha=0.15, color='#d9534f')
    ax5.set_title('Budget Sensitivity', fontweight='bold')
    ax5.set_ylabel('$ Billions')

# 6. Burn frequency
ax6 = fig.add_subplot(3, 3, 6)
if df1 is not None and 'prior_burns' in df1.columns:
    b = df1[df1['prior_burns'] >= 0]['prior_burns']
    sns.histplot(b, bins=6, discrete=True, ax=ax6, color='#f0ad4e')
    ax6.set_title('Burn Frequency Distribution', fontweight='bold')
    ax6.set_xlabel('Number of Burns')

# 7. Key metrics
ax7 = fig.add_subplot(3, 3, (7, 9))
ax7.axis('off')
stats = []
if df1 is not None:
    stats.append(f'Forest regions analyzed: {len(df1):,}')
    stats.append(f'Unplanned fire rate: {df1["unplanned_5"].mean()*100:.1f}%')
    if 'prior_burns' in df1.columns:
        stats.append(f'Avg prior burns: {df1["prior_burns"].mean():.1f}')
if plan:
    stats.append(f'Total budget: ${plan["total_budget_aud"]:,}')
    stats.append(f'Predicted fires: {plan["total_predicted_fires"]:,}')
if burn_mask is not None:
    stats.append(f'Burn scar: {burn_mask.mean()*100:.1f}% of scene')

ax7.text(0.02, 0.5, '\n'.join(stats), fontsize=14, va='center',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#dee2e6'))
ax7.set_title('Key Metrics', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Budget vs Fire Rate bubble chart
if df1 is not None and df2 is not None and state_cols:
    rates = {}
    for sc in state_cols:
        name = sc.replace('state_', '')
        sub = df1[df1[sc] == 1]
        if len(sub):
            rates[name] = sub['unplanned_5'].mean()

    comp = df2.copy()
    comp['fire_rate'] = comp['state'].map(rates)
    comp = comp.dropna()

    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(comp['fire_rate'], comp['budget'] / 1e9,
                        s=comp['fires'] * 2, c=range(len(comp)),
                        cmap='YlOrRd', alpha=0.7, edgecolors='black', linewidth=0.5)
    for _, r in comp.iterrows():
        ax.annotate(r['state'], (r['fire_rate'], r['budget'] / 1e9),
                   fontsize=10, ha='center', va='bottom')
    ax.set_xlabel('Fire Rate (2020-21)')
    ax.set_ylabel('Budget ($ Billions)')
    ax.set_title('Budget vs Fire Rate by State\n(Bubble size = predicted fire count)',
                fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Need both Project 1 and 2 data for comparison view')

In [ ]:
# Burn scar statistics
if burn_mask is not None and ndvi_diff is not None:
    from scipy import ndimage as ndi
    labeled, n_features = ndi.label(burn_mask)
    sizes = pd.Series([(labeled == i).sum() for i in range(1, n_features + 1)])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if len(sizes) > 0:
        sns.histplot(sizes, bins=30, ax=axes[0], color='#d9534f')
        axes[0].set_title(f'Burn Patch Size Distribution ({n_features} patches)',
                         fontweight='bold')
        axes[0].set_xlabel('Patch Size (pixels)')

    diff_burned = ndvi_diff[burn_mask]
    diff_unburned = ndvi_diff[~burn_mask]
    sns.histplot(diff_unburned, bins=50, alpha=0.5, label='Unburned',
                ax=axes[1], color='#2d6a2f')
    sns.histplot(diff_burned, bins=50, alpha=0.5, label='Burned',
                ax=axes[1], color='#d9534f')
    axes[1].set_title('NDVI Change: Burned vs Unburned Areas', fontweight='bold')
    axes[1].set_xlabel('NDVI Difference (Pre − Post)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('Run burn scar mapping first for this view')

In [ ]:
# Combined data table
if df1 is not None and df2 is not None and state_cols:
    summary = []
    for sc in state_cols:
        name = sc.replace('state_', '')
        sub = df1[df1[sc] == 1]
        if len(sub):
            ar = df2[df2['state'] == name]
            budget = ar['budget'].values[0] if len(ar) > 0 else 0
            summary.append({
                'State': name,
                'Regions': len(sub),
                'Fire Rate': f"{sub['unplanned_5'].mean()*100:.1f}%",
                'Avg Prior Burns': f"{sub['prior_burns'].mean():.2f}",
                'Budget ($M)': f"${budget/1e6:.0f}M" if budget else 'N/A'
            })
    pd.DataFrame(summary)
else:
    print('Run projects 1 and 2 first')